# More extensive EDA on dataset to see what needs claaning

In [0]:
df = spark.sql("SELECT * FROM marathos.bronze.raw_marathos")

df.display()

In [0]:
# Number of rows & columns

print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")

In [0]:
print(df.columns)

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
df.select("Event name").distinct().count()

In [0]:
df.select("Event name").distinct().display()
df.select("Year of event").distinct().display()
df.select("Athlete country").distinct().display()
df.select("Athlete gender").distinct().display()
df.select("Athlete club").distinct().display()
df.select("Event dates").distinct().display()

In [0]:
df.dtypes

In [0]:
df.select(
    [
        column
        for column, type_ in df.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()

In [0]:
df.summary().display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
df.select("Athlete year of birth").distinct().display()
df.select("Athlete age category").distinct().display()

In [0]:
df.display()

In [0]:
df.groupBy("Athlete age category").count().orderBy("count", ascending=False).display()

In [0]:
%sql
SELECT `Athlete year of birth`, COUNT(*) as num_athletes
FROM marathos.bronze.raw_marathos
GROUP BY `Athlete year of birth`
ORDER BY num_athletes DESC

In [0]:
country_counts = (
    df.groupBy("Athlete country")
    .count()
    .orderBy("count", ascending=False)
)

country_counts.limit(20).display()

## Cleaning 
- Validate status of event and preformance
    - if event has unit 'km' or 'mi¨' then preformance shold be in 'h'
    - if event has unit h then preformance should be in 'km'
    - remove all invalid rows (for simplicity you can consider d (days) as invalid and remove it)
- Convert time of athletes preformance to appropriate data type
- Create relevant ids
- Column names -> snakecase
- Nulls - fill with 'missing' or 'unknown'
- Weird characters 
- Round number of decimals
- Event date -> Timestamp

In [0]:
df.select("Event name", "Event distance/length", "Athlete performance").display()

In [0]:
df.select("Event distance/length").distinct().display()
df.select("Athlete performance").distinct().limit(100).display()

In [0]:
from pyspark.sql.functions import col, when

df_validated = df.withColumn(
    "is_valid",
    when(
        # distance-event -> performace should end with 'h'
        col("Event distance/length").rlike("km|mi|mile"),
        col("Athlete performance").endswith("h")
    ).when(
        # time-event -> performace should end with 'km'
        col("Event distance/length").rlike("h"),
        col("Athlete performance").endswith("km")
    ).when(
        col("Event distance/length").endswith("d"),
        False
    ).otherwise(False)
)

df_validated.groupBy("is_valid").count().display()

df_validated.filter(col("is_valid") == False).select(
    "Event distance/length", "Athlete performance"
).limit(20).display()